# Bibliotecas

In [5]:
!python --version

Python 3.10.20


In [6]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.version.cuda)
print('GPU     :', torch.cuda.get_device_name(0))
print('BF16    :', torch.cuda.is_bf16_supported())

PyTorch : 2.6.0+cu124
CUDA    : 12.4
GPU     : NVIDIA GeForce RTX 4090
BF16    : True


In [7]:
import numpy as np
import os
from datetime import datetime

import datasets
import evaluate

import torch
import torch.nn as nn

from transformers import Trainer, TrainerCallback, TrainingArguments, EarlyStoppingCallback
from transformers import AutoTokenizer, BertConfig, BertModel, BertPreTrainedModel

/home/guilhermelima/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Carregar os conjunto de dados pré-processado

In [8]:
#MAX_LENGTH = 512
NUM_TRAIN_EPOCHS = 10

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

RESULTS_DIRECTORY = './results/experiment_{}'.format(timestamp)

LOGGING_DIRECTORY = './logs/experiement_{}'.format(timestamp)

RESULTS_DIRECTORY, LOGGING_DIRECTORY

('./results/experiment_2026-06-11_10-43-41',
 './logs/experiement_2026-06-11_10-43-41')

In [9]:
#XPOS_LABELS = ['DET', 'NOUN', 'ADP', 'ADJ', 'VERB', 'CONJ', 'ADV', 'AUX', '.', 'PNOUN', 'NUM', 'PRON', 'PRT', 'ADPPRON', 'X']

#DEPREL_LABELS = ['det', 'attr', 'adpmod', 'amod', 'adpobj', 'ROOT', 'mark', 'nsubj', 'advmod', 'aux', 'adp', 'csubj', 'cc', 'conj', 'p', 'compmod', 'appos', 'acomp', 'num', 'nsubjpass', 'auxpass', 'dobj', 'poss', 'partmod', 'xcomp', 'ccomp', 'adpcomp', 'advcl', 'rcmod', 'nmod', 'neg', 'mwe', 'dep', 'parataxis', 'iobj', 'prt', 'infmod', 'csubjpass']

#UPOS_LABELS = ['DET', 'NOUN', 'ADP', 'ADJ', 'VERB', 'CONJ', 'ADV', '.', 'NUM', 'PRON', 'PRT', 'X']

UPOS_LABELS = ['DET', 'NOUN', 'VERB', 'PUNCT', 'SCONJ', 'ADP', 'ADJ', 'CCONJ', 'ADV', 'PROPN', 'AUX', 'NUM', 'PRON', 'SYM', 'X', 'INTJ']
DEPREL_LABELS = ['det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark', 'advcl', 'case', 'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod', 'flat:name', 'ccomp', 'cop', 'acl', 'nummod', 'acl:relcl', 'ccomp:speech', 'parataxis', 'csubj', 'aux:pass', 'appos', 'fixed', 'nsubj:pass', 'aux', 'nsubj:outer', 'obl:agent', 'expl:impers', 'expl', 'discourse', 'orphan', 'dislocated', 'flat', 'flat:foreign', 'iobj', 'vocative', 'csubj:outer', 'list', 'reparandum', 'csubj:pass']

DEPREL_LABELS_TO_IDX = {
    i : idx
    for idx, i in enumerate(DEPREL_LABELS)
}

IDX_TO_DEPREL_LABELS = {
    i: j
    for j, i in DEPREL_LABELS_TO_IDX.items()
}


UPOS_LABELS_TO_IDX = {
    i : idx
    for idx, i in enumerate(UPOS_LABELS)
}

IDX_TO_UPOS_LABELS = {
    i: j
    for j, i in UPOS_LABELS_TO_IDX.items()
}


#MAX_SEQUENCE_LENGTH = 512#
#PRETRAINED_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
#PRETRAINED_MODEL_NAME = "google-bert/bert-base-multilingual-cased"
NUM_TRAIN_EPOCHS = 10
PRETRAINED_MODEL_NAME = "amadeusai/modernJabuticaBERT-Base-1k"

In [10]:
print(len(UPOS_LABELS))

16


In [11]:
# initializing Config and Tokenizer
BERT_CONFIG = BertConfig.from_pretrained(PRETRAINED_MODEL_NAME)
TOKENIZER = AutoTokenizer.from_pretrained("amadeusai/modernJabuticaBERT-Base-1k", use_fast=True) #.is_fast

[transformers] You are using a model of type `modernbert` to instantiate a model of type `bert`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


In [12]:
BERT_CONFIG = {
    "vocab_size": 29794,
    "hidden_size": 768,
    "num_hidden_layers": 12,
    "num_attention_heads": 12,
    "intermediate_size": 3072,
    "hidden_act": "gelu",
    "max_position_embeddings": 512,
    "type_vocab_size": 2,
    "initializer_range": 0.02,
    "layer_norm_eps": 1e-12,
    "pad_token_id": 0,
    "attention_probs_dropout_prob": 0.1,
    "hidden_dropout_prob": 0.1,
}

In [13]:
import ast
import pandas as pd
import datasets
from datasets import Dataset, DatasetDict

def load_csv_as_hf_dataset(filepath):
    df = pd.read_csv(filepath)
    records = []
    for _, row in df.iterrows():
        records.append({
            'tokens': ast.literal_eval(row['tokens']),
            'upos': ast.literal_eval(row['upos']),
            'deprel': ast.literal_eval(row['deprel']),
            'head_tags': ast.literal_eval(str(row['head_tags'])),
            'deprel_tags': ast.literal_eval(str(row['deprel_tags'])),
            'upos_tags': ast.literal_eval(str(row['upos_tags'])),
        })
    return Dataset.from_list(records)

data = DatasetDict({
    'train': load_csv_as_hf_dataset('/home/guilhermelima/msc/data_dois/train_outxpos.csv'),
    'val':   load_csv_as_hf_dataset('/home/guilhermelima/msc/data_dois/val_outxpos.csv'),
    'test':  load_csv_as_hf_dataset('/home/guilhermelima/msc/data_dois/test_outxpos.csv'),
})
data

DatasetDict({
    train: Dataset({
        features: ['tokens', 'upos', 'deprel', 'head_tags', 'deprel_tags', 'upos_tags'],
        num_rows: 5893
    })
    val: Dataset({
        features: ['tokens', 'upos', 'deprel', 'head_tags', 'deprel_tags', 'upos_tags'],
        num_rows: 842
    })
    test: Dataset({
        features: ['tokens', 'upos', 'deprel', 'head_tags', 'deprel_tags', 'upos_tags'],
        num_rows: 1683
    })
})

# Modelo

In [14]:
# ============================================================
# dependencies.py  (ou célula do notebook)
# ============================================================
import torch
import torch.nn as nn
import numpy as np
from typing import Optional

from transformers import BertModel, BertPreTrainedModel


# ─────────────────────────────────────────────
# 1.  MLP não-linear com inicialização segura
# ─────────────────────────────────────────────
class MLP(nn.Module):
    """Projeção não-linear usada antes das camadas biaffine."""

    def __init__(self, in_features: int, out_features: int, dropout: float = 0.33):
        super().__init__()
        self.linear     = nn.Linear(in_features, out_features)
        self.activation = nn.ELU()
        self.norm       = nn.LayerNorm(out_features)  # normaliza entrada para biaffine
        self.dropout    = nn.Dropout(dropout)

        nn.init.orthogonal_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # LayerNorm garante σ≈1 antes de entrar na biaffine
        return self.dropout(self.norm(self.activation(self.linear(x))))


# ─────────────────────────────────────────────
# 2.  Biaffine  (Dozat & Manning, 2017)
# ─────────────────────────────────────────────
class Biaffine(nn.Module):
    """
    score[b, o, i, j] = Σ_h Σ_k  x[b,i,h] · W[o,h,k] · y[b,j,k]

    Convenção:
      x  → representação de DEPENDENTE   (token que recebe a aresta)
      y  → representação de HEAD         (token que emite a aresta)

    Arc scoring  : out_features=1,           bias_x=True,  bias_y=False
    Rel scoring  : out_features=num_deprel,  bias_x=True,  bias_y=True
    """

    def __init__(
        self,
        in_features:  int,
        out_features: int  = 1,
        bias_x:       bool = True,
        bias_y:       bool = True,
    ):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.bias_x       = bias_x
        self.bias_y       = bias_y

        self.weight = nn.Parameter(
            torch.zeros(
                out_features,
                in_features + int(bias_x),
                in_features + int(bias_y),
            )
        )
        # Escala correta para forma bilinear: Var(s)=H²·σ²_x·σ²_W·σ²_y=1
        # com std=1/H → Var(s)≈1 (evita saturação do softmax desde o step 0)
        nn.init.normal_(self.weight, std=1.0 / in_features)

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """
        x : [B, L, in_features]  — dependente
        y : [B, L, in_features]  — head candidato
        →   [B, out_features, L_dep, L_head]
        """
        if self.bias_x:
            x = torch.cat([x, x.new_ones(*x.shape[:-1], 1)], dim=-1)  # [B, L, H+1]
        if self.bias_y:
            y = torch.cat([y, y.new_ones(*y.shape[:-1], 1)], dim=-1)  # [B, L, H+1]

        # einsum: dep[b,i,h] · W[o,h,k] · head[b,j,k] → [b,o,i,j]
        return torch.einsum("bih,ohk,bjk->boij", x, self.weight, y)

In [15]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer


In [16]:
# ─────────────────────────────────────────────
# 3.  Modelo MTL
# ─────────────────────────────────────────────
class MultiTaskSentencePredictionEncoder(BertPreTrainedModel):
    """
    MTL para Universal Dependencies:
      • UPOS    → classificador linear sobre saída do encoder
      • HEAD    → biaffine arc  (out=1)
      • DEPREL  → biaffine rel  (out=num_deprel), avaliado na head predita
    """

    def __init__(
        self,
        config,
        num_deprel_labels: int,
        num_upos_labels:   int,
        arc_hidden:  int   = 500,
        rel_hidden:  int   = 100,
        mlp_dropout: float = 0.33,
    ):
        super().__init__(config)

        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels   = num_upos_labels
        self.arc_hidden        = arc_hidden
        self.rel_hidden        = rel_hidden

        # Persistir no config para recarregar checkpoints
        config.num_deprel_labels = num_deprel_labels
        config.num_upos_labels   = num_upos_labels
        config.arc_hidden        = arc_hidden
        config.rel_hidden        = rel_hidden

        self.bert = BertModel(config, add_pooling_layer=False)

        encoder_dropout = (
            config.classifier_dropout
            if getattr(config, "classifier_dropout", None) is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(encoder_dropout)

        # ── UPOS: linear direto sobre o encoder ──────────────────────
        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)
        nn.init.xavier_uniform_(self.upos_classifier.weight)
        nn.init.zeros_(self.upos_classifier.bias)

        # ── 4 MLPs: dois para arc, dois para rel ─────────────────────
        self.arc_head_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.arc_dep_mlp  = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.rel_dep_mlp  = MLP(config.hidden_size, rel_hidden, mlp_dropout)

        # ── Biaffines ────────────────────────────────────────────────
        # Arc: bias_y=False conforme Dozat & Manning 2017
        self.arc_biaffine = Biaffine(arc_hidden, out_features=1,
                                     bias_x=True, bias_y=False)
        # Rel: ambos os biases
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels,
                                     bias_x=True, bias_y=True)

        self.post_init()

    def _init_weights(self, module: nn.Module) -> None:
        """
        Estende o _init_weights do BERT para cobrir Biaffine (nn.Parameter direto).
        Chamado por apply() dentro de post_init() — e novamente após from_pretrained
        com _fast_init=False para garantir que nenhum parâmetro fique sem init.
        """
        if isinstance(module, Biaffine):
            # Escala correta para bilinear s = x^T W y: std = 1/H → Var(s) ≈ 1
            nn.init.normal_(module.weight, std=1.0 / module.in_features)
        elif isinstance(module, nn.Linear):
            # Cobre MLP.linear, upos_classifier e todas as camadas BERT
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    # ── forward ──────────────────────────────────────────────────────
    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,
        deprel_label:   Optional[torch.Tensor] = None,   # [B, L]
        upos_label:     Optional[torch.Tensor] = None,   # [B, L]
        head_label:     Optional[torch.Tensor] = None,   # [B, L]  valores ∈ {-100, 0..L-1}
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        seq = self.dropout(outputs.last_hidden_state)  # [B, L, H]
        B, L, _ = seq.shape

        # ── UPOS ─────────────────────────────────────────────────────
        logits_upos = self.upos_classifier(seq)          # [B, L, num_upos]

        # ── Arc scores ───────────────────────────────────────────────
        h_arc_dep  = self.arc_dep_mlp(seq)               # [B, L, arc_hidden]
        h_arc_head = self.arc_head_mlp(seq)
        # [B, 1, L_dep, L_head] → [B, L_dep, L_head]
        logits_head = self.arc_biaffine(h_arc_dep, h_arc_head).squeeze(1)

        # Guard: logits > 50 indicam corrupção CUDA ou explosão de gradiente
        # Com CUDA limpo e init correto, max esperado ≈ 6; nunca ultrapassa 50.
        if self.training and logits_head.abs().max() > 1e4:
            raise RuntimeError(
                f"logits_head.abs().max()={logits_head.abs().max():.3e} — "
                "contexto CUDA corrompido. Reinicie o kernel: Kernel → Restart → Run All Cells"
            )

        # Mascarar posições PAD como candidatos a HEAD (-1e4 em vez de -inf:
        # evita NaN em log_softmax quando a linha inteira seria -inf).
        if attention_mask is not None:
            pad_head_mask = (attention_mask == 0).unsqueeze(1)  # [B, 1, L_head]
            logits_head = logits_head.masked_fill(pad_head_mask, -1e4)

        # ── Rel scores ───────────────────────────────────────────────
        h_rel_dep  = self.rel_dep_mlp(seq)               # [B, L, rel_hidden]
        h_rel_head = self.rel_head_mlp(seq)
        # [B, num_deprel, L_dep, L_head]
        logits_rel = self.rel_biaffine(h_rel_dep, h_rel_head)

        # ── Selecionar logits de deprel na head predita ───────────────
        # Usado na saída (compute_metrics) — heads preditas, não gold
        # argmax sobre logits_head já mascarados → nunca retorna posição PAD
        arc_preds = logits_head.argmax(-1).clamp(0, L - 1)         # [B, L]
        idx_pred = (
            arc_preds
            .unsqueeze(-1).unsqueeze(-1)
            .expand(B, L, 1, self.num_deprel_labels)
        )
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()  # [B, L, L, num_deprel]
        logits_deprel_out = logits_rel_t.gather(2, idx_pred).squeeze(2)  # [B, L, num_deprel]

        # ── Loss (teacher forcing com gold heads) ─────────────────────
        loss = None
        if head_label is not None and deprel_label is not None and upos_label is not None:
            loss = self._compute_loss(
                logits_head, logits_rel, logits_upos,
                head_label, deprel_label, upos_label,
                B, L,
            )

        # Formato: (loss, deprel_logits, upos_logits, head_logits)
        # Compatível com o compute_metrics existente
        if loss is not None:
            return (loss, logits_deprel_out, logits_upos, logits_head)
        return (logits_deprel_out, logits_upos, logits_head)

    def _compute_loss(
        self,
        logits_head:  torch.Tensor,   # [B, L, L]
        logits_rel:   torch.Tensor,   # [B, num_deprel, L, L]
        logits_upos:  torch.Tensor,   # [B, L, num_upos]
        head_label:   torch.Tensor,   # [B, L]
        deprel_label: torch.Tensor,   # [B, L]
        upos_label:   torch.Tensor,   # [B, L]
        B: int,
        L: int,
    ) -> torch.Tensor:
        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

        # ── Sanitização: head OOB causa CUDA device-side assertion ─────────────
        # Frases truncadas podem ter HEAD apontando para posições além de L.
        # CrossEntropyLoss com target >= num_classes → NaN/crash em CUDA.
        oob_mask = (head_label != -100) & (head_label >= L)
        head_label_clean   = head_label.clone()
        deprel_label_clean = deprel_label.clone()
        head_label_clean[oob_mask]   = -100
        deprel_label_clean[oob_mask] = -100  # deprel sem head válido também invalida

        # ── HEAD: cross-entropy sobre L candidatos ────────────────────────────
        loss_head = loss_fct(
            logits_head.reshape(B * L, L),         # [B*L, L]
            head_label_clean.reshape(-1),           # [B*L] ∈ {-100, 0..L-1}
        )

        # ── DEPREL: teacher forcing com gold head ─────────────────────────────
        # Clamp duplo: gather exige índices em [0, L-1].
        # head_label_clean já tem -100 onde é inválido; clampar mapeia -100→0
        # (posição dummy, mas deprel_label_clean=-100 lá → ignorado pelo loss).
        safe_heads = head_label_clean.clamp(0, L - 1)              # [B, L]
        idx_gold = (
            safe_heads
            .unsqueeze(-1).unsqueeze(-1)
            .expand(B, L, 1, self.num_deprel_labels)
        )
        # .contiguous() antes de gather em tensor permutado
        logits_rel_t       = logits_rel.permute(0, 2, 3, 1).contiguous()  # [B, L, L, num_deprel]
        logits_deprel_gold = logits_rel_t.gather(2, idx_gold).squeeze(2)  # [B, L, num_deprel]

        loss_deprel = loss_fct(
            logits_deprel_gold.reshape(B * L, self.num_deprel_labels),
            deprel_label_clean.reshape(-1),
        )

        # ── UPOS ──────────────────────────────────────────────────────────────
        loss_upos = loss_fct(
            logits_upos.reshape(B * L, self.num_upos_labels),
            upos_label.reshape(-1),
        )

        # Detecta perda numericamente inválida antes de retornar
        # (evita propagar NaN/Inf que corrompem pesos silenciosamente)
        for name, val in [("loss_head", loss_head), ("loss_deprel", loss_deprel), ("loss_upos", loss_upos)]:
            if torch.isnan(val) or torch.isinf(val):
                raise RuntimeError(
                    f"{name}={val.item():.4e} — verifique head_label (range: "
                    f"{head_label_clean[head_label_clean != -100].tolist()[:5]}...) "
                    f"e logits (max={logits_head.abs().max().item():.3e})"
                )

        return loss_head + loss_deprel + loss_upos

In [17]:
from transformers import ModernBertModel, ModernBertPreTrainedModel

# ─────────────────────────────────────────────
# 3b.  Modelo MTL — ModernBERT
# ─────────────────────────────────────────────
class MultiTaskSentencePredictionEncoderModern(ModernBertPreTrainedModel):
    """
    MTL para Universal Dependencies (ModernBERT):
      • UPOS    → classificador linear sobre saída do encoder
      • HEAD    → biaffine arc  (out=1)
      • DEPREL  → biaffine rel  (out=num_deprel), avaliado na head predita

    Diferenças em relação à versão BERT:
      - Herda de ModernBertPreTrainedModel (base_model_prefix = "model")
      - Encoder armazenado como self.model para coincidir com as chaves do checkpoint
      - token_type_ids ignorado: ModernBERT não possui token-type embeddings
      - Fallback de dropout compatível com ModernBertConfig
    """

    def __init__(
        self,
        config,
        num_deprel_labels: int,
        num_upos_labels:   int,
        arc_hidden:  int   = 500,
        rel_hidden:  int   = 100,
        mlp_dropout: float = 0.33,
    ):
        super().__init__(config)

        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels   = num_upos_labels
        self.arc_hidden        = arc_hidden
        self.rel_hidden        = rel_hidden

        # Persistir no config para recarregar checkpoints
        config.num_deprel_labels = num_deprel_labels
        config.num_upos_labels   = num_upos_labels
        config.arc_hidden        = arc_hidden
        config.rel_hidden        = rel_hidden

        # "model" coincide com ModernBertPreTrainedModel.base_model_prefix
        self.model = ModernBertModel(config)

        # ModernBertConfig não tem hidden_dropout_prob; usa embedding_dropout como fallback
        encoder_dropout = (
            getattr(config, "classifier_dropout", None)
            or getattr(config, "hidden_dropout_prob", None)
            or getattr(config, "embedding_dropout", 0.1)
        )
        self.dropout = nn.Dropout(encoder_dropout)

        # ── UPOS: linear direto sobre o encoder ──────────────────────
        self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)
        nn.init.xavier_uniform_(self.upos_classifier.weight)
        nn.init.zeros_(self.upos_classifier.bias)

        # ── 4 MLPs: dois para arc, dois para rel ─────────────────────
        self.arc_head_mlp = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.arc_dep_mlp  = MLP(config.hidden_size, arc_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(config.hidden_size, rel_hidden, mlp_dropout)
        self.rel_dep_mlp  = MLP(config.hidden_size, rel_hidden, mlp_dropout)

        # ── Biaffines ────────────────────────────────────────────────
        self.arc_biaffine = Biaffine(arc_hidden, out_features=1,
                                     bias_x=True, bias_y=False)
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels,
                                     bias_x=True, bias_y=True)

        self.post_init()

    def _init_weights(self, module: nn.Module) -> None:
        """
        Estende o _init_weights do ModernBERT para cobrir Biaffine (nn.Parameter direto).
        """
        if isinstance(module, Biaffine):
            nn.init.normal_(module.weight, std=1.0 / module.in_features)
        elif isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    # ── forward ──────────────────────────────────────────────────────
    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,   # ignorado: ModernBERT não usa
        deprel_label:   Optional[torch.Tensor] = None,   # [B, L]
        upos_label:     Optional[torch.Tensor] = None,   # [B, L]
        head_label:     Optional[torch.Tensor] = None,   # [B, L]  valores ∈ {-100, 0..L-1}
    ):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            # token_type_ids não é suportado pelo ModernBERT
        )
        seq = self.dropout(outputs.last_hidden_state)  # [B, L, H]
        B, L, _ = seq.shape

        # ── UPOS ─────────────────────────────────────────────────────
        logits_upos = self.upos_classifier(seq)          # [B, L, num_upos]

        # ── Arc scores ───────────────────────────────────────────────
        h_arc_dep  = self.arc_dep_mlp(seq)               # [B, L, arc_hidden]
        h_arc_head = self.arc_head_mlp(seq)
        logits_head = self.arc_biaffine(h_arc_dep, h_arc_head).squeeze(1)

        if self.training and logits_head.abs().max() > 1e4:
            raise RuntimeError(
                f"logits_head.abs().max()={logits_head.abs().max():.3e} — "
                "contexto CUDA corrompido. Reinicie o kernel: Kernel → Restart → Run All Cells"
            )

        if attention_mask is not None:
            pad_head_mask = (attention_mask == 0).unsqueeze(1)  # [B, 1, L_head]
            logits_head = logits_head.masked_fill(pad_head_mask, -1e4)

        # ── Rel scores ───────────────────────────────────────────────
        h_rel_dep  = self.rel_dep_mlp(seq)               # [B, L, rel_hidden]
        h_rel_head = self.rel_head_mlp(seq)
        logits_rel = self.rel_biaffine(h_rel_dep, h_rel_head)  # [B, num_deprel, L, L]

        # ── Selecionar logits de deprel na head predita ───────────────
        arc_preds = logits_head.argmax(-1).clamp(0, L - 1)         # [B, L]
        idx_pred = (
            arc_preds
            .unsqueeze(-1).unsqueeze(-1)
            .expand(B, L, 1, self.num_deprel_labels)
        )
        logits_rel_t      = logits_rel.permute(0, 2, 3, 1).contiguous()  # [B, L, L, num_deprel]
        logits_deprel_out = logits_rel_t.gather(2, idx_pred).squeeze(2)  # [B, L, num_deprel]

        # ── Loss (teacher forcing com gold heads) ─────────────────────
        loss = None
        if head_label is not None and deprel_label is not None and upos_label is not None:
            loss = self._compute_loss(
                logits_head, logits_rel, logits_upos,
                head_label, deprel_label, upos_label,
                B, L,
            )

        if loss is not None:
            return (loss, logits_deprel_out, logits_upos, logits_head)
        return (logits_deprel_out, logits_upos, logits_head)

    def _compute_loss(
        self,
        logits_head:  torch.Tensor,
        logits_rel:   torch.Tensor,
        logits_upos:  torch.Tensor,
        head_label:   torch.Tensor,
        deprel_label: torch.Tensor,
        upos_label:   torch.Tensor,
        B: int,
        L: int,
    ) -> torch.Tensor:
        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

        oob_mask = (head_label != -100) & (head_label >= L)
        head_label_clean   = head_label.clone()
        deprel_label_clean = deprel_label.clone()
        head_label_clean[oob_mask]   = -100
        deprel_label_clean[oob_mask] = -100

        loss_head = loss_fct(
            logits_head.reshape(B * L, L),
            head_label_clean.reshape(-1),
        )

        safe_heads = head_label_clean.clamp(0, L - 1)
        idx_gold = (
            safe_heads
            .unsqueeze(-1).unsqueeze(-1)
            .expand(B, L, 1, self.num_deprel_labels)
        )
        logits_rel_t       = logits_rel.permute(0, 2, 3, 1).contiguous()
        logits_deprel_gold = logits_rel_t.gather(2, idx_gold).squeeze(2)

        loss_deprel = loss_fct(
            logits_deprel_gold.reshape(B * L, self.num_deprel_labels),
            deprel_label_clean.reshape(-1),
        )

        loss_upos = loss_fct(
            logits_upos.reshape(B * L, self.num_upos_labels),
            upos_label.reshape(-1),
        )

        for name, val in [("loss_head", loss_head), ("loss_deprel", loss_deprel), ("loss_upos", loss_upos)]:
            if torch.isnan(val) or torch.isinf(val):
                raise RuntimeError(
                    f"{name}={val.item():.4e} — verifique head_label (range: "
                    f"{head_label_clean[head_label_clean != -100].tolist()[:5]}...) "
                    f"e logits (max={logits_head.abs().max().item():.3e})"
                )

        return loss_head + loss_deprel + loss_upos


In [18]:
# ─────────────────────────────────────────────
# 4.  Fábrica de modelos MTL
# ─────────────────────────────────────────────
def build_model(
    name_model:        str,
    num_deprel_labels: int,
    num_upos_labels:   int,
    arc_hidden:  int   = 500,
    rel_hidden:  int   = 100,
    mlp_dropout: float = 0.33,
):
    """
    Instancia a classe MTL correta de acordo com a arquitetura do modelo pretrained.

    model_type == 'modernbert'  →  MultiTaskSentencePredictionEncoderModern
    qualquer outro (bert, ...)  →  MultiTaskSentencePredictionEncoder
    """
    from transformers import AutoConfig

    config = AutoConfig.from_pretrained(name_model)

    if config.model_type == "modernbert":
        cls = MultiTaskSentencePredictionEncoderModern
    else:
        cls = MultiTaskSentencePredictionEncoder

    print(f"[build_model] model_type='{config.model_type}' → {cls.__name__}")

    return cls.from_pretrained(
        name_model,
        config=config,
        num_deprel_labels=num_deprel_labels,
        num_upos_labels=num_upos_labels,
        arc_hidden=arc_hidden,
        rel_hidden=rel_hidden,
        mlp_dropout=mlp_dropout,
        _fast_init=False,
    ).to("cuda" if torch.cuda.is_available() else "cpu")


# Tokenização

In [19]:
class POSDataset:

    def __init__(self, tokenizer_ckpt):
        #self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_ckpt, add_prefix_space=True)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_ckpt)
        
    def align_labels_with_tokens(self, labels, word_ids):
        new_labels = []
        current_word = None
        for word_id in word_ids:
            if word_id != current_word:
                # Start of a new word!
                current_word = word_id
                try:
                    label = -100 if word_id is None else labels[word_id]
                except:
                    label = -100
                new_labels.append(label)
            elif word_id is None:
                # Special token
                new_labels.append(-100)
            else:
                # Same word as previous token
                label = labels[word_id]
                # If the label is B-XXX we change it to I-XXX
                #if label % 2 == 1:
                #    label += 1
                new_labels.append(label)

        return new_labels
    def preprocess_function(self, examples):
        tokenized_inputs = self.tokenizer(
        examples["tokens"], truncation=True , padding="max_length" , is_split_into_words=True, max_length=512)

        
        '''all_labels_xpos = examples["xpos_tags"]
        new_labels_xpos = []
        for i, labels in enumerate(all_labels_xpos):
            word_ids = tokenized_inputs.word_ids(i)
            new_labels_xpos.append(self.align_labels_with_tokens(labels, word_ids))'''

        all_labels_deprel = examples["deprel_tags"]
        new_labels_deprel = []
        for i, labels in enumerate(all_labels_deprel):
            word_ids = tokenized_inputs.word_ids(i)
            new_labels_deprel.append(self.align_labels_with_tokens(labels, word_ids))
        
        all_labels_upos = examples["upos_tags"]
        new_labels_upos = []
        for i, labels in enumerate(all_labels_upos):
            word_ids = tokenized_inputs.word_ids(i)
            new_labels_upos.append(self.align_labels_with_tokens(labels, word_ids))

        all_labels_head = examples["head_tags"]
        new_labels_head = []
        for i, labels in enumerate(all_labels_head):
            word_ids = tokenized_inputs.word_ids(i)
            new_labels_head.append(self.align_labels_with_tokens(labels, word_ids))

        #tokenized_inputs["xpos_label"] = new_labels_xpos
        tokenized_inputs["deprel_label"] = new_labels_deprel
        tokenized_inputs["upos_label"] = new_labels_upos
        tokenized_inputs["head_label"] = new_labels_head
        return tokenized_inputs

    def create_data(self, train, test):

        tokenized_train_dataset = train.map(
            self.preprocess_function,
            batched=True,
            remove_columns=train.column_names
        )

        tokenized_test_dataset = test.map(
            self.preprocess_function,
            batched=True,
            remove_columns= test.column_names
        )

        return tokenized_train_dataset, tokenized_test_dataset

In [20]:
#nerdataset = POSDataset("neuralmind/bert-base-portuguese-cased")
#nerdataset = POSDataset("google-bert/bert-base-multilingual-cased")
#nerdataset = POSDataset("neuralmind/bert-large-portuguese-cased")
nerdataset = POSDataset("amadeusai/modernJabuticaBERT-Base-1k")

In [21]:
train_data, valid_data = nerdataset.create_data(data['train'], data['val'])

Map: 100%|██████████| 842/842 [00:00<00:00, 2200.65 examples/s]


In [22]:
train_data

Dataset({
    features: ['input_ids', 'attention_mask', 'deprel_label', 'upos_label', 'head_label'],
    num_rows: 5893
})

# Data Collator

In [23]:
def data_collator(batch, padding_token_id=TOKENIZER.pad_token_id):
    input_ids = [item["input_ids"] for item in batch]
    attention_masks = [item["attention_mask"] for item in batch]
    #xpos_label = [item["xpos_label"] for item in batch]
    deprel_label = [item["deprel_label"] for item in batch]
    upos_label = [item["upos_label"] for item in batch]
    head_label  = [item["head_label"] for item in batch]
    
    
    max_len = max(len(ids) for ids in input_ids)
    '''
    for i, labels in enumerate(head_label):  # ou head_label, deprel_label
        for j, label in enumerate(labels):
            if label != -100 and (label < 0 or label >= 75):
                print(f"Erro no exemplo {i}, posição {j}: label={label}")'''

    input_ids = torch.tensor([ids + [padding_token_id] * (max_len - len(ids)) for ids in input_ids])
    attention_masks = torch.tensor([masks + [0] * (max_len - len(masks)) for masks in attention_masks])
    #xpos_label = torch.tensor([labels + [-100] * (max_len - len(labels)) for labels in xpos_label])
    deprel_label = torch.tensor([labels + [-100] * (max_len - len(labels)) for labels in deprel_label])
    upos_label = torch.tensor([labels + [-100] * (max_len - len(labels)) for labels in upos_label])
    head_label = torch.tensor([labels + [-100] * (max_len - len(labels)) for labels in head_label])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        #"xpos_label": xpos_label,
        "deprel_label": deprel_label,
        "upos_label": upos_label,
        "head_label": head_label
    }

# Criação do Modelo

In [24]:
#model = MultiTaskSentencePrediction.from_pretrained(
#    PRETRAINED_MODEL_NAME,
#    config=BERT_CONFIG,
#    num_deprel_labels=len(DEPREL_LABELS), num_upos_labels=len(UPOS_LABELS), num_head_labels=100
#)
from transformers import AutoConfig


config = AutoConfig.from_pretrained(PRETRAINED_MODEL_NAME)

# ESSENCIAL: registrar a classe no config
#MultiTaskSentencePrediction.config_class = config.__class_

In [25]:
"""model = MultiTaskSentencePredictionEncoder.from_pretrained(
    PRETRAINED_MODEL_NAME,
    config=config,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS),
    _fast_init=False,  # evita NaN em arc_biaffine.weight (torch.empty neste sistema)
)"""
#model = model.to("cpu")
#torch.cuda.empty_cache()

'model = MultiTaskSentencePredictionEncoder.from_pretrained(\n    PRETRAINED_MODEL_NAME,\n    config=config,\n    num_deprel_labels=len(DEPREL_LABELS),\n    num_upos_labels=len(UPOS_LABELS),\n    _fast_init=False,  # evita NaN em arc_biaffine.weight (torch.empty neste sistema)\n)'

In [26]:
config = AutoConfig.from_pretrained(PRETRAINED_MODEL_NAME)

model = build_model(
    PRETRAINED_MODEL_NAME,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS)
)


[build_model] model_type='modernbert' → MultiTaskSentencePredictionEncoderModern


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 4168.45it/s]
[transformers] MultiTaskSentencePredictionEncoderModern LOAD REPORT from: amadeusai/modernJabuticaBERT-Base-1k
Key                        | Status  | 
---------------------------+---------+-
arc_dep_mlp.norm.weight    | MISSING | 
arc_head_mlp.linear.bias   | MISSING | 
arc_dep_mlp.linear.bias    | MISSING | 
upos_classifier.weight     | MISSING | 
rel_head_mlp.norm.weight   | MISSING | 
rel_dep_mlp.linear.weight  | MISSING | 
arc_biaffine.weight        | MISSING | 
rel_dep_mlp.norm.bias      | MISSING | 
arc_dep_mlp.linear.weight  | MISSING | 
rel_head_mlp.linear.bias   | MISSING | 
arc_head_mlp.norm.bias     | MISSING | 
arc_dep_mlp.norm.bias      | MISSING | 
rel_biaffine.weight        | MISSING | 
rel_dep_mlp.linear.bias    | MISSING | 
rel_head_mlp.linear.weight | MISSING | 
arc_head_mlp.linear.weight | MISSING | 
upos_classifier.bias       | MISSING | 
rel_head_mlp.norm.bias     | MISSING | 
arc_head_mlp.norm.we

In [27]:
len(DEPREL_LABELS)

44

In [28]:
len(UPOS_LABELS)

16

In [29]:
model

MultiTaskSentencePredictionEncoderModern(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
  

# Compute Metrics Old

In [30]:
"""import numpy as np


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    deprel_logits, upos_logits, head_logits = logits
    deprel_labels, upos_labels, head_labels = labels

    # Argmax
    deprel_preds = np.argmax(deprel_logits, axis=-1)
    upos_preds   = np.argmax(upos_logits, axis=-1)
    head_preds   = np.argmax(head_logits, axis=-1)

    # Máscara válida: token com HEAD e DEPREL anotados
    valid_mask = (
        (head_labels != -100) &
        (deprel_labels != -100)
    )

    # Filtrar
    head_preds = head_preds[valid_mask]
    head_labels = head_labels[valid_mask]

    deprel_preds = deprel_preds[valid_mask]
    deprel_labels = deprel_labels[valid_mask]

    # UAS: HEAD correto
    uas = (head_preds == head_labels).mean()

    # LAS: HEAD + DEPREL corretos
    las = ((head_preds == head_labels) &
           (deprel_preds == deprel_labels)).mean()

    # Métricas auxiliares (opcional)
    upos_mask = upos_labels != -100
    upos_acc = (upos_preds[upos_mask] == upos_labels[upos_mask]).mean()

    return {
        "uas": float(uas),
        "las": float(las),
        "upos_accuracy": float(upos_acc),
    }
"""


'import numpy as np\n\n\ndef compute_metrics(eval_pred):\n    logits, labels = eval_pred\n\n    deprel_logits, upos_logits, head_logits = logits\n    deprel_labels, upos_labels, head_labels = labels\n\n    # Argmax\n    deprel_preds = np.argmax(deprel_logits, axis=-1)\n    upos_preds   = np.argmax(upos_logits, axis=-1)\n    head_preds   = np.argmax(head_logits, axis=-1)\n\n    # Máscara válida: token com HEAD e DEPREL anotados\n    valid_mask = (\n        (head_labels != -100) &\n        (deprel_labels != -100)\n    )\n\n    # Filtrar\n    head_preds = head_preds[valid_mask]\n    head_labels = head_labels[valid_mask]\n\n    deprel_preds = deprel_preds[valid_mask]\n    deprel_labels = deprel_labels[valid_mask]\n\n    # UAS: HEAD correto\n    uas = (head_preds == head_labels).mean()\n\n    # LAS: HEAD + DEPREL corretos\n    las = ((head_preds == head_labels) &\n           (deprel_preds == deprel_labels)).mean()\n\n    # Métricas auxiliares (opcional)\n    upos_mask = upos_labels != -100\

In [31]:
import numpy as np
import json
import os

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


def compute_metrics(eval_pred, PRETRAINED_MODEL=None, FOLD=None, TRIAL=None):

    save_path = "epoch_predictions/predictions_results.json"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # ---------------- IDENTIDADE DO EXPERIMENTO ----------------
    model_name = PRETRAINED_MODEL.split("/")[-1] if PRETRAINED_MODEL else "unknown_model"
    run_id = f"{model_name}_fold{FOLD}_trial{TRIAL}"

    logits, labels = eval_pred
    deprel_logits, upos_logits, head_logits = logits
    deprel_labels, upos_labels, head_labels = labels

    # ---------------- PROBABILIDADES ----------------
    deprel_probs = softmax(deprel_logits, axis=-1)
    upos_probs   = softmax(upos_logits, axis=-1)
    head_probs   = softmax(head_logits, axis=-1)

    # ---------------- PREDIÇÕES ----------------
    deprel_preds = np.argmax(deprel_logits, axis=-1)
    upos_preds   = np.argmax(upos_logits, axis=-1)
    head_preds   = np.argmax(head_logits, axis=-1)

    # ---------------- MASK DEPENDENCY ----------------
    valid_mask = (
        (head_labels != -100) &
        (deprel_labels != -100)
    )

    head_preds_masked   = head_preds[valid_mask]
    head_labels_masked  = head_labels[valid_mask]
    head_probs_masked   = head_probs[valid_mask]

    deprel_preds_masked  = deprel_preds[valid_mask]
    deprel_labels_masked = deprel_labels[valid_mask]
    deprel_probs_masked  = deprel_probs[valid_mask]

    # ---------------- MÉTRICAS PRINCIPAIS ----------------
    uas = (head_preds_masked == head_labels_masked).mean()

    las = (
        (head_preds_masked == head_labels_masked) &
        (deprel_preds_masked == deprel_labels_masked)
    ).mean()

    # ---------------- UPOS ----------------
    upos_mask = upos_labels != -100

    upos_preds_masked  = upos_preds[upos_mask]
    upos_labels_masked = upos_labels[upos_mask]
    upos_probs_masked  = upos_probs[upos_mask]

    upos_acc = (upos_preds_masked == upos_labels_masked).mean()

    # ---------------- CARREGAR JSON ----------------
    if os.path.exists(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            all_data = json.load(f)
    else:
        all_data = {}

    # ---------------- META ----------------
    if "META" not in all_data:
        all_data["META"] = {
            "model_name": model_name,
            "pretrained_model": PRETRAINED_MODEL,
            "fold": FOLD,
            "trial": TRIAL,
            "run_id": run_id
        }

    # ---------------- STORAGE ----------------
    if "RANK_PREDICTIONS" not in all_data:
        all_data["RANK_PREDICTIONS"] = {
            "deprel": [],
            "upos": [],
            "head": []
        }

    def get_correct_rank(prob_vector, correct_label):
        sorted_indices = np.argsort(prob_vector)[::-1]
        return int(np.where(sorted_indices == correct_label)[0][0] + 1)

    # ---------------- DEPREL ----------------
    for i in range(len(deprel_preds_masked)):
        probs = deprel_probs_masked[i]
        correct_label = int(deprel_labels_masked[i])
        pred_label = int(deprel_preds_masked[i])

        all_data["RANK_PREDICTIONS"]["deprel"].append({
            "run_id": run_id,
            "model": model_name,
            "fold": FOLD,
            "trial": TRIAL,
            "index": i,
            "correct_label": correct_label,
            "pred_label": pred_label,
            "correct_prob": float(probs[correct_label]),
            "pred_prob": float(probs[pred_label]),
            "correct_rank": get_correct_rank(probs, correct_label)
        })

    # ---------------- UPOS ----------------
    for i in range(len(upos_preds_masked)):
        probs = upos_probs_masked[i]
        correct_label = int(upos_labels_masked[i])
        pred_label = int(upos_preds_masked[i])

        all_data["RANK_PREDICTIONS"]["upos"].append({
            "run_id": run_id,
            "model": model_name,
            "fold": FOLD,
            "trial": TRIAL,
            "index": i,
            "correct_label": correct_label,
            "pred_label": pred_label,
            "correct_prob": float(probs[correct_label]),
            "pred_prob": float(probs[pred_label]),
            "correct_rank": get_correct_rank(probs, correct_label)
        })

    # ---------------- HEAD ----------------
    for i in range(len(head_preds_masked)):
        probs = head_probs_masked[i]
        correct_label = int(head_labels_masked[i])
        pred_label = int(head_preds_masked[i])

        all_data["RANK_PREDICTIONS"]["head"].append({
            "run_id": run_id,
            "model": model_name,
            "fold": FOLD,
            "trial": TRIAL,
            "index": i,
            "correct_label": correct_label,
            "pred_label": pred_label,
            "correct_prob": float(probs[correct_label]),
            "pred_prob": float(probs[pred_label]),
            "correct_rank": get_correct_rank(probs, correct_label)
        })

    # ---------------- SALVAR ----------------
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(all_data, f, indent=2, ensure_ascii=False)

    return {
        "uas": float(uas),
        "las": float(las),
        "upos_accuracy": float(upos_acc),
    }

In [32]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

True
12.4
NVIDIA GeForce RTX 4090


In [33]:
from sklearn.model_selection import KFold
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

In [34]:
from datasets import concatenate_datasets



In [35]:
import torch

if torch.cuda.is_available():
    print("GPU está ativa!")
    print(f"Nome da GPU: {torch.cuda.get_device_name(0)}")
    print(f"Número de GPUs disponíveis: {torch.cuda.device_count()}")
else:
    print("GPU não está disponível ou não foi detectada.")


GPU está ativa!
Nome da GPU: NVIDIA GeForce RTX 4090
Número de GPUs disponíveis: 1


In [36]:
import os
os.environ["WANDB_API_KEY"] = "7499f50c2fffad3ce3e34b7b7b02152988a1742c"

import wandb
#wandb.login()


In [37]:
output_directory = RESULTS_DIRECTORY
evaluation_strategy = 'epoch'
gradint_accumulation_steps = 1
learning_rate = 4.796780552717818e-05
weight_decay = 0.01
max_grad_norm = 1
num_train_epochs = 10 #NUM_TRAIN_EPOCHS
lr_scheduler_type = 'linear'
warmup_ratio = 0.05
logging_dir = LOGGING_DIRECTORY
logging_strategy = 'epoch'
save_strategy = "epoch"
save_total_limit = 1
#label_names = ['xpos_label', 'deprel_label', 'upos_label', 'head_label']
label_names = ['deprel_label', 'upos_label', 'head_label']
load_best_model_at_end = False
metric_for_best_model="las"
greater_is_better = True
label_smoothing_factor = 0
#report_to = 'tensorboard'
gradient_checkpointing = False
remove_unused_columns=False

In [38]:
import subprocess
import sys


def reinstalar_pytorch_nightly():
    """
    Remove torch/torchvision e instala as versões nightly CUDA 12.8.
    """

    comandos = [
        [
            sys.executable,
            "-m",
            "pip",
            "uninstall",
            "-y",
            "torch",
            "torchvision",
        ],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--pre",
            "torch",
            "torchvision",
            "torchaudio",
            "--index-url",
            "https://download.pytorch.org/whl/nightly/cu128",
        ],
    ]

    for cmd in comandos:
        print(f"\nExecutando: {' '.join(cmd)}\n")
        subprocess.run(cmd, check=True)

    print("\nInstalação concluída com sucesso.")


# Exemplo de uso


# Optuna

In [39]:
# ══════════════════════════════════════════════════════════════════════════════
# DIAGNÓSTICO DE ESTADO CUDA — executar SEMPRE antes de iniciar treino
# ══════════════════════════════════════════════════════════════════════════════
import os
import torch

# Força execução síncrona: CUDA device-side assertions aparecem imediatamente
# em vez de corromperam o contexto silenciosamente.
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Verifica se o contexto CUDA está limpo (sem corrupção de run anterior)
def _check_cuda_context():
    if not torch.cuda.is_available():
        return
    try:
        t = torch.tensor([1.0], device="cuda")
        result = (t * 2).item()
        assert result == 2.0, f"CUDA context corrompido: {result}"
        del t
        torch.cuda.empty_cache()
        print("✓ Contexto CUDA limpo")
    except Exception as e:
        raise RuntimeError(
            f"Contexto CUDA corrompido: {e}\n"
            "→ Reinicie o kernel (Kernel > Restart) e execute as células novamente."
        )

_check_cuda_context()


✓ Contexto CUDA limpo


In [40]:
import optuna
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score
import os
import json
import gc
import torch
#os.environ["WANDB_DISABLED"] = "true"

from datasets import concatenate_datasets

def save_result_to_json(result_dict, filename="results.jsonl"):
    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(result_dict, ensure_ascii=False) + "\n")



# K Folds

def objective(trial, name_model, train_data, valid_data, tokenizer=None, data_collator=None, name="model_run"):
    # Hiperparâmetros sugeridos pelo Optuna
    #while True:
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.1, 0.3)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.3, 0.5)
    num_train_epochs = trial.suggest_int("num_train_epochs", 40, 40)
    label_smoothing_factor = 0 #trial.suggest_float("label_smoothing_factor", 0.0, 0.2)
    #config = (weight_decay, learning_rate, warmup_ratio)
    
    import os
    os.environ["WANDB_API_KEY"] = "7499f50c2fffad3ce3e34b7b7b02152988a1742c"

    import wandb
    

    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    # Suponha que seus dados estejam assim:
    # data = {'train': list de exemplos, 'val': list de exemplos}

    # 1. Juntar tudo em um único data
    #full_data = concatenate_datasets([train_data, valid_data])
    full_data = concatenate_datasets([data['train'], data['val']])
    # Converter para numpy array só para facilitar a indexação
    indices = np.arange(len(full_data))
    nerdataset = POSDataset(name_model)
    for fold, (train_idx, val_idx) in enumerate(kf.split(full_data)):
        #gc.collect()

        #torch.cuda.empty_cache()

        #torch.cuda.ipc_collect()

        
        # reinstalar_pytorch_nightly()
        wandb.init(
            entity="gdlima-universidade-federal-de-pelotas",
            project="hf-optuna",
            name=f"biaffine_{name_model}_trial{trial.number}_fold{fold}",  # <- aqui define o nome do run
            config={
                "learning_rate": learning_rate,
                "architecture": name_model,
                "epochs": num_train_epochs,
                "weight_decay": weight_decay,
                "warmup_ratio": warmup_ratio,
                "label_smoothing_factor": label_smoothing_factor
            },
        )
        config = AutoConfig.from_pretrained(name_model)

        model = MultiTaskSentencePredictionEncoder.from_pretrained(
            name_model,
            config=config,
            num_deprel_labels=len(DEPREL_LABELS),
            num_upos_labels=len(UPOS_LABELS),
            _fast_init=False,  # garante que _init_weights roda para Biaffine/MLP
        )

        # Verificar CPU antes de mover para GPU — distingue bug de init vs CUDA corrompido
        for _name, _p in model.named_parameters():
            if not _p.requires_grad:
                continue
            if torch.isnan(_p).any() or torch.isinf(_p).any():
                raise RuntimeError(
                    f"[CPU] Peso '{_name}' é NaN/Inf antes de mover para CUDA. "
                    "Bug de inicialização — revise _init_weights."
                )

        _device = "cuda" if torch.cuda.is_available() else "cpu"
        model = model.to(_device)

        # Verificar CUDA: se o peso ficou NaN/Inf só após .to("cuda") → contexto corrompido
        for _name, _p in model.named_parameters():
            if not _p.requires_grad:
                continue
            if torch.isnan(_p).any() or torch.isinf(_p).any():
                raise RuntimeError(
                    f"[CUDA] Peso '{_name}' tornou-se NaN/Inf após .to(cuda). "
                    "Contexto CUDA corrompido → Kernel → Restart → Run All Cells."
                )
        
        print(f"\n===== Fold {fold + 1} / {k} =====")

        # 3. Selecionar os dados (Dataset, não list)
        train_split = full_data.select(train_idx.tolist())
        val_split = full_data.select(val_idx.tolist())
        
        # 4. Tokenizar novamente usando sua função existente
        train_data, valid_data = nerdataset.create_data(train_split, val_split)
        

        #train_data = train_data.shuffle(seed=42).select(range(400))
        #valid_data = valid_data.shuffle(seed=42).select(range(200))
        training_args = TrainingArguments(
            # output_dir por fold: evita conflito no load_best_model_at_end
            output_dir=f"./parser_{name_model.replace('/', '_')}/fold_{fold}",
            fp16=False,
            bf16=False,  # desativar até estabilizar; reativar após confirmar convergência
            eval_strategy="epoch",
            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,   # TODO: migrar para warmup_steps
            max_grad_norm=1.0,  # biaffine com 3 losses somados: 1.0 é seguro
            lr_scheduler_type="linear",
            logging_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            load_best_model_at_end=True,  # necessário para EarlyStoppingCallback funcionar
            metric_for_best_model="las",
            greater_is_better=True,
            label_smoothing_factor=0.0,
            gradient_checkpointing=False, # incompatível com biaffine em alguns casos
            remove_unused_columns=False,  # obrigatório — modelo recebe labels customizadas
            label_names=["deprel_label", "upos_label", "head_label"],
            report_to="wandb",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            dataloader_num_workers=4,
        )

        early_stop_callback = EarlyStoppingCallback(5)
        # Inicializa o Trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_data,
            eval_dataset=valid_data,
            #tokenizer=tokenizer,          # opcional
            compute_metrics=compute_metrics,
            data_collator=data_collator,  # opcional
            callbacks=[early_stop_callback]  # se você quiser usar callbacks
        )


        trainer.train()
        eval_result = trainer.evaluate()

        result_dict = {
            "name": name_model,
            "Fold": fold+1,
            "trial_number": trial.number,
            "hyperparameters": {
                "learning_rate": learning_rate,
                "num_train_epochs": num_train_epochs,
                "weight_decay": weight_decay,
                "warmup_ratio": warmup_ratio,
                "label_smoothing_factor": label_smoothing_factor
            },
            "las": eval_result["eval_las"],
        }

        save_result_to_json(result_dict)
        print(f"Fold {fold + 1} metrics:", eval_result)
            

In [41]:

'''def objective(trial, name_model, train_data, valid_data, tokenizer=None, data_collator=None, optuna_count=0, name="model_run"):
    # Hiperparâmetros sugeridos pelo Optuna
    forbidden_configs = {
    (0.2147035642975132, 1.7109988776595992e-05, 0.41601400434863894),
    (0.2147035642975132, 2.208391537977642e-05, 0.41601400434863894),
    (0.2147035642975132, 2.5771715479732614e-05, 0.41601400434863894),
    (0.2147035642975132, 2.3346341563131348e-05, 0.41601400434863894),
    (0.2147035642975132, 4.818415253407452e-05, 0.41601400434863894),
}
    while True:
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
        weight_decay = trial.suggest_float("weight_decay", 0.1, 0.3)
        warmup_ratio = trial.suggest_float("warmup_ratio", 0.3, 0.5)
        num_train_epochs = trial.suggest_int("num_train_epochs", 40, 40)
        config = (weight_decay, learning_rate, warmup_ratio)

        if config not ilabel_smoothing_factorn forbidden_configs:
            break  # valor válido


    import os
    os.environ["WANDB_API_KEY"] = "7499f50c2fffad3ce3e34b7b7b02152988a1742c"

    import wandb
    
    #wandb.init(project="bracis", entity="gdlima")
    run = wandb.init(
        # Set the wandb entity where your project will be logged (generally your team name).
        entity="gdlima-universidade-federal-de-pelotas",
        # Set the wandb project where this run will be logged.
        project="bracis",
        # Track hyperparameters and run metadata.
        config={
            "learning_rate": learning_rate,
            "architecture": name_model,
            "epochs": num_train_epochs,
            "weight_decay": weight_decay,
            "warmup_ratio": warmup_ratio,
        },
    )
    #kf = KFold(n_splits=k, shuffle=True, random_state=42)

    # Suponha que seus dados estejam assim:
    # data = {'train': list de exemplos, 'val': list de exemplos}

    # 1. Juntar tudo em um único data
    #full_data = concatenate_datasets([train_data, valid_data])
    #full_data = concatenate_datasets([data['train'], data['val']])
    # Converter para numpy array só para facilitar a indexação
    #indices = np.arange(len(full_data))

    #for fold, (train_idx, val_idx) in enumerate(kf.split(full_data)):


    config = AutoConfig.from_pretrained(name_model)

    model = MultiTaskSentencePredictionEncoder.from_pretrained(
            name_model,
            config=config,
            num_deprel_labels=len(DEPREL_LABELS),
            num_upos_labels=len(UPOS_LABELS)
        ).to("cuda" if torch.cuda.is_available() else "cpu")
        
        #print(f"\n===== Fold {fold + 1} / {k} =====")

        # 3. Selecionar os dados (Dataset, não list)
        #train_split = full_data.select(train_idx.tolist())
        #val_split = full_data.select(val_idx.tolist())

        # 4. Tokenizar novamente usando sua função existente    fp16=False,
    bf16=False,
    #train_data, valid_data = nerdataset.create_data(train_split, val_split)


        # Setup training arguments
    training_args = TrainingArguments(
            #output_dir= f'./ettin-decoder-150m_parser',
            eval_strategy=evaluation_strategy,
            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,
            weight_decay=weight_decay,
            logging_dir=logging_dir,
            label_names=label_names,
            max_grad_norm=max_grad_norm,
            lr_scheduler_type=lr_scheduler_type,
            warmup_ratio=warmup_ratio,
            logging_strategy=logging_strategy,
            save_strategy=save_strategy,
            save_total_limit=save_total_limit,
            #load_best_model_at_end=load_best_model_at_end,
            metric_for_best_model=metric_for_best_model,
            greater_is_better=greater_is_better,
            label_smoothing_factor=label_smoothing_factor,
            #report_to=report_to,
            #per_device_train_batch_size=per_device_batch_size,
            gradient_checkpointing=gradient_checkpointing,
            remove_unused_columns=remove_unused_columns,
        )

        
        # Inicializa o Trainer
    trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_data,  # Limitando o treinamento para 10 exemplos
            eval_dataset=valid_data,   # Limitando a avaliação para 10 exemplos
            #tokenizer=tokenizer,          # opcional
            compute_metrics=compute_metrics,
            data_collator=data_collator,  # opcional
            # callbacks=[early_stop_callback]  # se você quiser usar callbacks
        )


    trainer.train()
    eval_result = trainer.evaluate()
    print(eval_result)
    result_dict = {
            "name": name_model,
            "Fold": optuna_count,
            "trial_number": trial.number,
            "hyperparameters": {
                "learning_rate": learning_rate,
                "num_train_epochs": num_train_epochs,
            },
            "las": eval_result["eval_las"],
        }

    save_result_to_json(result_dict)
    print(f"Optuna {optuna_count + 1} metrics:", eval_result)
    '''

'def objective(trial, name_model, train_data, valid_data, tokenizer=None, data_collator=None, optuna_count=0, name="model_run"):\n    # Hiperparâmetros sugeridos pelo Optuna\n    forbidden_configs = {\n    (0.2147035642975132, 1.7109988776595992e-05, 0.41601400434863894),\n    (0.2147035642975132, 2.208391537977642e-05, 0.41601400434863894),\n    (0.2147035642975132, 2.5771715479732614e-05, 0.41601400434863894),\n    (0.2147035642975132, 2.3346341563131348e-05, 0.41601400434863894),\n    (0.2147035642975132, 4.818415253407452e-05, 0.41601400434863894),\n}\n    while True:\n        learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)\n        weight_decay = trial.suggest_float("weight_decay", 0.1, 0.3)\n        warmup_ratio = trial.suggest_float("warmup_ratio", 0.3, 0.5)\n        num_train_epochs = trial.suggest_int("num_train_epochs", 40, 40)\n        config = (weight_decay, learning_rate, warmup_ratio)\n\n        if config not ilabel_smoothing_factorn forbidden_c

In [42]:
# Crear estudio y optimizar
study = optuna.create_study(direction="maximize")



[I 2026-06-11 10:43:49,944] A new study created in memory with name: no-name-8f012d6d-69d9-41ef-8441-b55a6374559d


In [43]:
'''#Decoder
import torch
import gc

def limpar_gpu():
    gc.collect()                                # Limpa lixo da CPU
    torch.cuda.empty_cache()                    # Libera cache da GPU
    torch.cuda.ipc_collect()                    # Coleta memória interprocessos
    torch.cuda.reset_peak_memory_stats()        # Reseta estatísticas de memória
    torch.cuda.synchronize()                    # Garante execução sincronizada

# Exemplo de uso antes do treinamento
if torch.cuda.is_available():
    limpar_gpu()'''

'#Decoder\nimport torch\nimport gc\n\ndef limpar_gpu():\n    gc.collect()                                # Limpa lixo da CPU\n    torch.cuda.empty_cache()                    # Libera cache da GPU\n    torch.cuda.ipc_collect()                    # Coleta memória interprocessos\n    torch.cuda.reset_peak_memory_stats()        # Reseta estatísticas de memória\n    torch.cuda.synchronize()                    # Garante execução sincronizada\n\n# Exemplo de uso antes do treinamento\nif torch.cuda.is_available():\n    limpar_gpu()'

In [44]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [45]:
'''models_decoder = ["maritaca-ai/sabia-7b" , "nicholasKluge/TeenyTinyLlama-460m", "TucanoBR/Tucano-630m", "https://huggingface.co/adalbertojunior/bart-base-portuguese"]


for model_name in models_decoder:
# Otimização
    study.optimize(
        lambda trial: objective(trial, MultiTaskSentencePrediction.from_pretrained(
        model_name,
        config=config,
        num_deprel_labels=len(DEPREL_LABELS),
        num_upos_labels=len(UPOS_LABELS)
    ).to(device), train_data, valid_data,  data_collator),
        n_trials=5
    )'''



'models_decoder = ["maritaca-ai/sabia-7b" , "nicholasKluge/TeenyTinyLlama-460m", "TucanoBR/Tucano-630m", "https://huggingface.co/adalbertojunior/bart-base-portuguese"]\n\n\nfor model_name in models_decoder:\n# Otimização\n    study.optimize(\n        lambda trial: objective(trial, MultiTaskSentencePrediction.from_pretrained(\n        model_name,\n        config=config,\n        num_deprel_labels=len(DEPREL_LABELS),\n        num_upos_labels=len(UPOS_LABELS)\n    ).to(device), train_data, valid_data,  data_collator),\n        n_trials=5\n    )'

In [46]:
from transformers import AutoModel, AutoConfig
from transformers import BertConfig

"""
def build_model(model_name, num_deprel_labels, num_upos_labels):

    config = AutoConfig.from_pretrained(model_name)
    
    model = MultiTaskSentencePrediction.from_pretrained(
    model_name,
    config=config,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS)
)

    return model"""

'\ndef build_model(model_name, num_deprel_labels, num_upos_labels):\n\n    config = AutoConfig.from_pretrained(model_name)\n    \n    model = MultiTaskSentencePrediction.from_pretrained(\n    model_name,\n    config=config,\n    num_deprel_labels=len(DEPREL_LABELS),\n    num_upos_labels=len(UPOS_LABELS)\n)\n\n    return model'

In [47]:
!pip install numpy==1.24.2


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [48]:
!pip install protobuf==3.20.3

  Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (679 bytes)
Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (1.1 MB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.0
    Uninstalling protobuf-7.35.0:
      Successfully uninstalled protobuf-7.35.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.2 requires protobuf!=5.28.0,!=5.29.0,<8,>4.21.0, but you have protobuf 3.20.3 which is incompatible.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [49]:
!export PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python

In [50]:
!export WANDB_NOTEBOOK_NAME="optuna_models"

In [51]:
!pip install --upgrade wandb

  Using cached protobuf-7.35.0-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached protobuf-7.35.0-cp310-abi3-manylinux2014_x86_64.whl (327 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [52]:
#!pip uninstall -y torch torchvision

In [53]:
# instala uma vez só
#!pip uninstall --pre torch torchvision torchaudio \
#--index-url https://download.pytorch.org/whl/nightly/cu128

In [54]:
from transformers import set_seed

set_seed(42)

In [55]:
import os
import random
import numpy as np
import torch

def seed_everything(seed: int = 42):
    # Python
    random.seed(seed)
    
    # Numpy
    np.random.seed(seed)
    
    # PyTorch (CPU)
    torch.manual_seed(seed)
    
    # PyTorch (GPU)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Determinismo (importante!)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Algumas libs usam isso
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    # Para transformers (às vezes ajuda)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    print(f"Seed definida como {seed}")

# Uso
seed_everything(42)

Seed definida como 42


In [56]:
"""#models_encoder = ["neuralmind/bert-large-portuguese-cased"]

models_encoder = ["neuralmind/bert-base-portuguese-cased", "google-bert/bert-base-multilingual-cased", "neuralmind/bert-large-portuguese-cased"]
#models_encoder = ["google-bert/bert-base-multilingual-cased"] #, "neuralmind/bert-large-portuguese-cased"]
#models_encoder = ["google-bert/bert-base-multilingual-cased", "neuralmind/bert-large-portuguese-cased" , "distilbert/distilbert-base-uncased", "google-bert/bert-large-uncased"]

optuna_count = 0
#print(len(train_data))

#train_data = train_data.shuffle(seed=42).select(range(1000))
#valid_data = valid_data.shuffle(seed=42).select(range(1000))
for model_name in models_encoder:   

    study = optuna.create_study(direction="maximize")

    print("="*50)
    print(f'Modelo: {model_name}')
    print("="*50)

    study.optimize(
        lambda trial: objective(
            trial,
            model_name,
            train_data,
            valid_data,
            data_collator,
            #optuna_count=optuna_count  # Passando o valor atual de optuna_count
        ),
        n_trials=10
    )

    # Incrementar optuna_count após a execução de cada trial
    optuna_count += 1"""

'#models_encoder = ["neuralmind/bert-large-portuguese-cased"]\n\nmodels_encoder = ["neuralmind/bert-base-portuguese-cased", "google-bert/bert-base-multilingual-cased", "neuralmind/bert-large-portuguese-cased"]\n#models_encoder = ["google-bert/bert-base-multilingual-cased"] #, "neuralmind/bert-large-portuguese-cased"]\n#models_encoder = ["google-bert/bert-base-multilingual-cased", "neuralmind/bert-large-portuguese-cased" , "distilbert/distilbert-base-uncased", "google-bert/bert-large-uncased"]\n\noptuna_count = 0\n#print(len(train_data))\n\n#train_data = train_data.shuffle(seed=42).select(range(1000))\n#valid_data = valid_data.shuffle(seed=42).select(range(1000))\nfor model_name in models_encoder:   \n\n    study = optuna.create_study(direction="maximize")\n\n    print("="*50)\n    print(f\'Modelo: {model_name}\')\n    print("="*50)\n\n    study.optimize(\n        lambda trial: objective(\n            trial,\n            model_name,\n            train_data,\n            valid_data,\n   

In [57]:
"""print("Melhor Modelo:", study.best_value)
print("Melhor Hiperparametros:", study.best_params)"""

'print("Melhor Modelo:", study.best_value)\nprint("Melhor Hiperparametros:", study.best_params)'

# K-Folds

In [58]:
'''from sklearn.model_selection import KFold
import numpy as np
from datasets import concatenate_datasets
# Suponha que seus dados estejam assim:
# data = {'train': list de exemplos, 'val': list de exemplos}

# 1. Juntar tudo em um único data
full_data = concatenate_datasets([data['train'], data['val']])
# 2. Criar os índices para K-Fold
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

# Converter para numpy array só para facilitar a indexação
indices = np.arange(len(full_data))

for fold, (train_idx, val_idx) in enumerate(kf.split(full_data)):
    print(f"\n===== Fold {fold + 1} / {k} =====")

    # 3. Selecionar os dados (Dataset, não list)
    train_split = full_data.select(train_idx.tolist())
    val_split = full_data.select(val_idx.tolist())

    # 4. Tokenizar novamente usando sua função existente
    train_data, valid_data = nerdataset.create_data(train_split, val_split)

    # 5. (Re)criar o modelo (importante para que cada fold comece do zero)
    #model = model_init()  # define essa função para criar um novo modelo
    model = MultiTaskSentencePrediction.from_pretrained(
    PRETRAINED_MODEL_NAME,
    config=config,
    num_deprel_labels=len(DEPREL_LABELS), num_upos_labels=len(UPOS_LABELS), num_head_labels=100
    )
    # 6. Criar o Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=valid_data,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
        # tokenizer=tokenizer,  # se necessário
        # callbacks=[early_stop_callback]  # se estiver usando
    )

    # 7. Treinar
    trainer.train()

    # 8. Avaliar
    metrics = trainer.evaluate()
    print(f"Fold {fold + 1} metrics:", metrics)
'''

'from sklearn.model_selection import KFold\nimport numpy as np\nfrom datasets import concatenate_datasets\n# Suponha que seus dados estejam assim:\n# data = {\'train\': list de exemplos, \'val\': list de exemplos}\n\n# 1. Juntar tudo em um único data\nfull_data = concatenate_datasets([data[\'train\'], data[\'val\']])\n# 2. Criar os índices para K-Fold\nk = 5\nkf = KFold(n_splits=k, shuffle=True, random_state=42)\n\n# Converter para numpy array só para facilitar a indexação\nindices = np.arange(len(full_data))\n\nfor fold, (train_idx, val_idx) in enumerate(kf.split(full_data)):\n    print(f"\n===== Fold {fold + 1} / {k} =====")\n\n    # 3. Selecionar os dados (Dataset, não list)\n    train_split = full_data.select(train_idx.tolist())\n    val_split = full_data.select(val_idx.tolist())\n\n    # 4. Tokenizar novamente usando sua função existente\n    train_data, valid_data = nerdataset.create_data(train_split, val_split)\n\n    # 5. (Re)criar o modelo (importante para que cada fold comece

# Treinamento Modelo

In [59]:
"""output_directory = RESULTS_DIRECTORY
evaluation_strategy = 'epoch'
gradint_accumulation_steps = 1
learning_rate = 1.3460913609009724e-05
weight_decay = 0.25845496044756133
max_grad_norm = 1
num_train_epochs = 40 #NUM_TRAIN_EPOCHS
lr_scheduler_type = 'linear'
warmup_ratio = 0.40558421095582453
logging_dir = LOGGING_DIRECTORY
logging_strategy = 'epoch'
save_strategy = 'epoch'
save_total_limit = 1
#label_names = ['xpos_label', 'deprel_label', 'upos_label', 'head_label']
label_names = ['deprel_label', 'upos_label', 'head_label']
load_best_model_at_end = False
metric_for_best_model="las"
greater_is_better = True
label_smoothing_factor = 0
#report_to = 'tensorboard'
gradient_checkpointing = False
remove_unused_columns=False"""

'output_directory = RESULTS_DIRECTORY\nevaluation_strategy = \'epoch\'\ngradint_accumulation_steps = 1\nlearning_rate = 1.3460913609009724e-05\nweight_decay = 0.25845496044756133\nmax_grad_norm = 1\nnum_train_epochs = 40 #NUM_TRAIN_EPOCHS\nlr_scheduler_type = \'linear\'\nwarmup_ratio = 0.40558421095582453\nlogging_dir = LOGGING_DIRECTORY\nlogging_strategy = \'epoch\'\nsave_strategy = \'epoch\'\nsave_total_limit = 1\n#label_names = [\'xpos_label\', \'deprel_label\', \'upos_label\', \'head_label\']\nlabel_names = [\'deprel_label\', \'upos_label\', \'head_label\']\nload_best_model_at_end = False\nmetric_for_best_model="las"\ngreater_is_better = True\nlabel_smoothing_factor = 0\n#report_to = \'tensorboard\'\ngradient_checkpointing = False\nremove_unused_columns=False'

In [60]:
train_data, valid_data = nerdataset.create_data(data['train'], data['val'])

Map: 100%|██████████| 842/842 [00:00<00:00, 1959.49 examples/s]


In [61]:
import torch

print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability())

x = torch.randn(2, 2).cuda()
#print(x)

2.6.0+cu124
NVIDIA GeForce RTX 4090
(8, 9)


In [62]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

True
12.4
NVIDIA GeForce RTX 4090


In [63]:
"""wandb.init(
            entity="gdlima-universidade-federal-de-pelotas",
            project="hf-optuna",
            name=f"{models_encoder[0]}_best_trial",
            config={
                "learning_rate": learning_rate,
                "architecture": models_encoder[0],
                "epochs": num_train_epochs,
                "weight_decay": weight_decay,
                "warmup_ratio": warmup_ratio,
            },
        )
config = AutoConfig.from_pretrained(models_encoder[0])

model = MultiTaskSentencePredictionEncoder.from_pretrained(
            models_encoder[0],
            config=config,
            num_deprel_labels=len(DEPREL_LABELS),
            num_upos_labels=len(UPOS_LABELS)
        ).to("cuda" if torch.cuda.is_available() else "cpu")

training_args = TrainingArguments(
            #output_dir= f'./ettin-decoder-150m_parser',
            eval_strategy=evaluation_strategy,
            learning_rate=learning_rate,
            num_train_epochs=40,
            weight_decay=weight_decay,
            logging_dir=logging_dir,
            label_names=label_names,
            max_grad_norm=max_grad_norm,
            lr_scheduler_type=lr_scheduler_type,
            warmup_ratio=warmup_ratio,
            logging_strategy=logging_strategy,
            save_strategy=save_strategy,
            save_total_limit=save_total_limit,
            #load_best_model_at_end=load_best_model_at_end,
            metric_for_best_model=metric_for_best_model,
            greater_is_better=greater_is_better,
            label_smoothing_factor=label_smoothing_factor,
            #report_to=report_to,
            #per_device_train_batch_size=per_device_batch_size,
            gradient_checkpointing=gradient_checkpointing,
            remove_unused_columns=remove_unused_columns,
            report_to="wandb"
        )

#early_stop_callback = EarlyStoppingCallback(3)
        # Inicializa o Trainer

"""


'wandb.init(\n            entity="gdlima-universidade-federal-de-pelotas",\n            project="hf-optuna",\n            name=f"{models_encoder[0]}_best_trial",\n            config={\n                "learning_rate": learning_rate,\n                "architecture": models_encoder[0],\n                "epochs": num_train_epochs,\n                "weight_decay": weight_decay,\n                "warmup_ratio": warmup_ratio,\n            },\n        )\nconfig = AutoConfig.from_pretrained(models_encoder[0])\n\nmodel = MultiTaskSentencePredictionEncoder.from_pretrained(\n            models_encoder[0],\n            config=config,\n            num_deprel_labels=len(DEPREL_LABELS),\n            num_upos_labels=len(UPOS_LABELS)\n        ).to("cuda" if torch.cuda.is_available() else "cpu")\n\ntraining_args = TrainingArguments(\n            #output_dir= f\'./ettin-decoder-150m_parser\',\n            eval_strategy=evaluation_strategy,\n            learning_rate=learning_rate,\n            num_train_

In [64]:
"""trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_data,
            eval_dataset=valid_data,
            #tokenizer=tokenizer,          # opcional
            compute_metrics=lambda p: compute_metrics(
                                            p,
                                            PRETRAINED_MODEL=PRETRAINED_MODEL_NAME,
                                            FOLD=1000,
                                            TRIAL=1000
                                        ),
            data_collator=data_collator,  # opcional
            #callbacks=[early_stop_callback]  # se você quiser usar callbacks
        )"""

'trainer = Trainer(\n            model=model,\n            args=training_args,\n            train_dataset=train_data,\n            eval_dataset=valid_data,\n            #tokenizer=tokenizer,          # opcional\n            compute_metrics=lambda p: compute_metrics(\n                                            p,\n                                            PRETRAINED_MODEL=PRETRAINED_MODEL_NAME,\n                                            FOLD=1000,\n                                            TRIAL=1000\n                                        ),\n            data_collator=data_collator,  # opcional\n            #callbacks=[early_stop_callback]  # se você quiser usar callbacks\n        )'

In [65]:
"""trainer.train()"""

'trainer.train()'

In [66]:
import torch
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

12.4
NVIDIA GeForce RTX 4090
(8, 9)


In [67]:
#!pip uninstall -y torch torchvision


In [68]:
#!pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128

In [69]:
"""eval_result = trainer.evaluate()
print(f"Best Trial metrics:", eval_result)"""

'eval_result = trainer.evaluate()\nprint(f"Best Trial metrics:", eval_result)'

In [70]:
# Setup training arguments
training_args = TrainingArguments(
    #output_dir= f'./ettin-decoder-150m_parser',
    eval_strategy=evaluation_strategy,
    learning_rate=3.504036658120871e-05,
    num_train_epochs=40,
    weight_decay=0.1583486838649066,
    logging_dir=logging_dir,
    label_names=label_names,
    max_grad_norm=max_grad_norm,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=0.30279577856576295,
    logging_strategy=logging_strategy,
    save_strategy=save_strategy,
    save_total_limit=save_total_limit,
    load_best_model_at_end=load_best_model_at_end,
    metric_for_best_model=metric_for_best_model,
    greater_is_better=greater_is_better,
    label_smoothing_factor=label_smoothing_factor,
    report_to="wandb",
    gradient_checkpointing=gradient_checkpointing
)
#early_stop_callback = EarlyStoppingCallback(3)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [71]:
run = wandb.init(
        # Set the wandb entity where your project will be logged (generally your team name).
        entity="gdlima-universidade-federal-de-pelotas",
        name=f"jabuticabert-1k-Biaffine-Selecionado",  # <- aqui define o nome do run
        # Set the wandb project where this run will be logged.
        project="hf-optuna",
        # Track hyperparameters and run metadata.
        config={
            "learning_rate": 3.504036658120871e-05,
            "architecture": "jabuticabert-1k-Biaffine-Selecionado",
            "epochs": 40,
            "weight_decay": 0.1583486838649066,
            "warmup_ratio": 0.30279577856576295,
        },
    )


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: gdlima (gdlima-universidade-federal-de-pelotas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [72]:
model

MultiTaskSentencePredictionEncoderModern(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
  

In [73]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    # tokenizer=tokenizer,
    compute_metrics=lambda p: compute_metrics(
                                            p,
                                            PRETRAINED_MODEL=PRETRAINED_MODEL_NAME,
                                            FOLD=1000,
                                            TRIAL=1000
                                        ),
    data_collator=data_collator,
    
    #callbacks=[early_stop_callback]
)


In [74]:
trainer.train()

Epoch,Training Loss,Validation Loss,Uas,Las,Upos Accuracy
1,9.354714,6.127495,0.146571,0.104955,0.613947
2,5.221494,3.597127,0.262242,0.228849,0.885742
3,3.473840,2.674963,0.337708,0.311418,0.944428
4,2.671645,2.197135,0.402625,0.378120,0.960917
5,2.151891,1.899414,0.464219,0.441126,0.969141
6,1.731539,1.634654,0.540557,0.517756,0.971716
7,1.359810,1.370629,0.644889,0.621672,0.973211
8,1.051941,1.913915,0.472858,0.455372,0.974000
9,0.837061,1.112549,0.744736,0.720979,0.978070
10,0.690763,1.086161,0.769406,0.744362,0.980022


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]


TrainOutput(global_step=29480, training_loss=0.7981783478057045, metrics={'train_runtime': 5347.0594, 'train_samples_per_second': 44.084, 'train_steps_per_second': 5.513, 'total_flos': 8.107995303051264e+16, 'train_loss': 0.7981783478057045, 'epoch': 40.0})

In [75]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Uas,Las,Upos Accuracy
0.001049,2.287051,40,0.880093,0.862068,0.985380


{'eval_loss': 2.287051200866699,
 'eval_uas': 0.880093034846534,
 'eval_las': 0.8620675333305644,
 'eval_upos_accuracy': 0.9853802384017942}

In [77]:
model

MultiTaskSentencePredictionEncoderModern(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
  

In [7]:
import json
import pandas as pd
from pathlib import Path

# ======================================================
# CAMINHO DO ARQUIVO JSONL
# (1 JSON por linha)
# ======================================================

json_file = "results_jabuticabert_1k_linear.jsonl"

# ======================================================
# LEITURA DO ARQUIVO
# ======================================================

records = []

with open(json_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        # ignora linhas vazias
        if not line:
            continue

        obj = json.loads(line)

        row = {
            "model": obj["name"],
            "fold": obj["Fold"],
            "trial": obj["trial_number"],
            "las": obj["las"],
        }

        # adiciona hiperparâmetros
        row.update(obj["hyperparameters"])

        records.append(row)

# ======================================================
# DATAFRAME
# ======================================================

df = pd.DataFrame(records)

print("\n================ DADOS CARREGADOS ================\n")
print(df.head())

# ======================================================
# CROSS VALIDATION
# ======================================================

group_cols = [
    "model",
    "trial",
    "learning_rate",
    "num_train_epochs",
    "weight_decay",
    "warmup_ratio",
]

cv_results = (
    df.groupby(group_cols)
    .agg(
        mean_las=("las", "mean"),
        std_las=("las", "std"),
        min_las=("las", "min"),
        max_las=("las", "max"),
        folds=("las", "count"),
    )
    .reset_index()
)

# ordena pelo melhor LAS médio
cv_results = cv_results.sort_values(
    by="mean_las",
    ascending=False
)

# ======================================================
# MELHOR HIPERPARÂMETRO
# ======================================================

best = cv_results.iloc[0]

print("\n================ MELHOR HIPERPARÂMETRO ================\n")

print(f"Modelo: {best['model']}")
print(f"Trial: {best['trial']}")

print("\nHiperparâmetros:")
print(f"  learning_rate    = {best['learning_rate']}")
print(f"  num_train_epochs = {best['num_train_epochs']}")
print(f"  weight_decay     = {best['weight_decay']}")
print(f"  warmup_ratio     = {best['warmup_ratio']}")

print("\nResultados Cross Validation:")
print(f"  Mean LAS = {best['mean_las']:.6f}")
print(f"  Std LAS  = {best['std_las']:.6f}")
print(f"  Min LAS  = {best['min_las']:.6f}")
print(f"  Max LAS  = {best['max_las']:.6f}")
print(f"  Folds    = {best['folds']}")

# ======================================================
# SALVAR RANKING COMPLETO
# ======================================================

output_csv = "cv_results_jabuticabert_1k_linear.csv"

cv_results.to_csv(output_csv, index=False)

print(f"\nRanking salvo em: {output_csv}")

# ======================================================
# TOP 10
# ======================================================

print("\n================ TOP 10 ================\n")

print(
    cv_results[
        [
            "model",
            "trial",
            "mean_las",
            "std_las",
            "learning_rate",
            "weight_decay",
            "warmup_ratio",
        ]
    ]
    .head(10)
    .to_string(index=False)
)


================ DADOS CARREGADOS ================

                                  model  fold  trial       las  learning_rate  \
0  amadeusai/modernJabuticaBERT-Base-1k     1      0  0.729226       0.000022   
1  amadeusai/modernJabuticaBERT-Base-1k     2      0  0.722991       0.000022   
2  amadeusai/modernJabuticaBERT-Base-1k     3      0  0.743455       0.000022   
3  amadeusai/modernJabuticaBERT-Base-1k     4      0  0.724164       0.000022   
4  amadeusai/modernJabuticaBERT-Base-1k     5      0  0.712214       0.000022   

   num_train_epochs  weight_decay  warmup_ratio  label_smoothing_factor  
0                40      0.227242      0.359989                       0  
1                40      0.227242      0.359989                       0  
2                40      0.227242      0.359989                       0  
3                40      0.227242      0.359989                       0  
4                40      0.227242      0.359989                       0  

===============